In [1]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 5, 7, 11])

nb_name = "01_assembling_anndata.ipynb"
_compare_version(nb_name)

# 01 - Assembling or loading anndata object
<hr style="border:2px solid black"> </hr>

## 1 - Description
This notebook is dedicated to load or create an anndata object. The anndata object is prepared for the following analysis notebooks and finally stored as an `.h5ad` file. Based on the available data files there are multiple options to create the anndata object.

### Available options:
#### 1. `.h5ad` file
Choose this option if you have one or more `.h5ad` file(s). The file could be provided by a preprocessing pipeline, a public dataset or a preceeding analysis.

#### 2. star solo quant folder
This option is intended to assemble anndata object from the standard [star solo](https://github.com/alexdobin/STAR/tree/master) output folder (`quant/`). This is done by scaning through the folder structure and using the `*_matrix.mtx`, `*_barcodes.tsv` and `*_genes.tsv` to create an anndata object per sample. The sample anndata objects are finally combined.

#### 3. .mtx, barcode.tsv, genes.tsv
Choose this option if you have the expression matrix in `.mtx` format, a file containing the barcodes (`*_barcodes.tsv`) and a file containing the genes (`*_genes.tsv`). Use this option for cases with the aforementioned three files available e.g. from a public dataset.

#### 4. convert from R object
This option should be used if the data was processed using 'R'. This can either be a `.rds` or `.robj` file.

______

## 2 - Setup

In [2]:
import pandas as pd
import sctoolbox
import sctoolbox.utils as utils
import sctoolbox.utils.assemblers as assembler

sctoolbox.settings.settings_from_config("config.yaml", key="01")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(utils.general.get_version_report(report="versions.yml"))

/mnt/workspace2/hschult/.conda/sctoolbox/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Created directory: ../logs
Created directory: ../adatas/
Created directory: ../report/01_assembly/


,Name,Version
0,Python,"3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:25:55) [GCC 14.3.0]"
1,pandas,2.3.3
2,sctoolbox,NA
3,numpy,2.3.5
4,scanpy,1.11.5
5,anndata,0.12.6


___

## 3 - Read in data
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [3]:
# For option 1: The path to an existing .h5ad file.
# A list or dict for multiple files e.g. {"rep_1": "1.h5ad", "rep_2": "2.h5ad"}.
path_h5ad = {
    "rep_1": "../../../data/RNA/GSM5402430_2dpf_rep_1.h5ad"
}

# For option 2: Path to a star solo quant directory
path_quant = ""

# For option 3: Directory containing .mtx, barcodes.tsv and genes.tsv
path_mtx = ""

# For option 4: This is the path to the Seurat (.rds, .robj) file
# A list or dict for multiple files e.g. {"rep_1": "1.rds", "rep_2": "2.rds"}.
path_rds = ""

# Optional: Provide name of Seurat layer to be stored in anndata.
# When reading multiple Seurat files set to string to get the same layer from every object
# or set to list or dict to get specified layers from each object.
# In the second case it must be the same datatype as path_rds.
layer = None

batch_label = None  # Name of the column added to adata.obs only if Option 1 or 4 above is a dict.

-------

In [4]:
if sum(map(lambda x: x != "", [path_h5ad, path_quant, path_mtx, path_rds])) != 1:
    del path_h5ad, path_quant, path_mtx, path_rds
    raise ValueError("Please set only one of the above variables. Adjust the cell above and re-run.")

____

### 3.1 - Option 1: Read from h5ad

In [5]:
if path_h5ad:
    adata = assembler.from_h5ad(path_h5ad, label=batch_label, report="01_assembly.png")

___

### 3.2 -  Option 2: Assemble from preprocessing pipeline 'quant' folder

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [6]:
# change to None if your genes.tsv and bracodes.tsv don't have a header
header = 'infer'

# Set up additional sample the information below.
# Follows the scheme:
# <sample name>:<type>:<value>
# E.g.: 
# sample1:condition:room_air
# sample1:timepoint:early
# sample2:timepoint:late
the_10X_yml = []

__________

In [7]:
if path_quant:
    adata = assembler.from_quant(path_quant, the_10X_yml, header=header, report="01_assembly.png")

____

### 3.3 - Option 3: Create an anndata object from .mtx, barcodes.tsv and genes.tsv

In [8]:
# adjust in case of different naming schemes for any of the input files

mtx = '*matrix.mtx*'  # pattern for the file that contains counts
barcodes = '*barcodes.tsv*'  # pattern for the file that contains barcode information
variables = '*genes.tsv*'  # pattern for the file that contains variable information

In [9]:
if path_mtx:
    adata = assembler.from_mtx(path_mtx, mtx=mtx, barcodes=barcodes, variables=variables, report="01_assembly.png")

___

### 3.4 - Option 4: Convert from Seurat to anndata object

In [10]:
# Converting from Seurat to anndata object
if path_rds:
    adata = assembler.from_R(rds_file=path_rds, label=batch_label, layer=layer, report="01_assembly.png")

__________

## 4 - Prepare anndata
Rename or remove observation (.obs) and variable (.var) columns and layers as needed.

The AnnData object contains all information regarding the dataset at hand. It is commonly stored as a `.h5ad` file. As it is central to this single cell analysis here is a broad overview of its key attributes:

Attributes are accessed with `AnnData.<attribute_name>` e.g. `adata.obs`.

| Attribute | Description |
|-----------|-------------|
|`obs`|A table (pandas DataFrame) containing information to each observation. Observations often refer to individual barcodes or wells and are interpreted as cells.|
|`var`|A table (pandas DataFrame) containing information to each variable. Variables are the measurements taken for each cell and are e.g. genes (RNA) or peaks (ATAC).|
|`X`|A (sparse) matrix containing the actual measurement values. It has the shape `obs x var`. It is the main matrix used for calculations and contains read counts interpreted as gene expression (RNA) or chromatin openness (ATAC). Depending on the analysis progress this may be normalized or corrected.|
|`layers`|A dict-like structure containing additional named matrices. This can be used to store different versions of the matrix e.g. 'raw' and 'normalized'.|
|`raw`|A special attribute containing a raw representation of the dataset. A number of Scanpy functions use this implicitly when available, thus we recommend storing the raw data within the `layers` to avoid confusion.|

For more information visit the [AnnData Read the Docs](https://anndata.readthedocs.io/en/latest/index.html).

In [11]:
import pandas as pd

with pd.option_context('display.max_rows', 5,'display.max_columns', None):
    display(adata.obs)
    display(adata.var)
    display(f"AnnData.raw: {'Yes' if adata.raw else 'No'}")
    display(f"AnnData layers: {list(adata.layers.keys())}")

,paper_cluster,filename,sample,timepoint,batch
AAACCTGAGATGCGAC-1,8,GSM5402430_scRNAseq_Sox10_Cre_bact_BtR_2dpf_re...,GSM5402430,2dpf,rep_1
AAACCTGAGCGCTCCA-1,13,GSM5402430_scRNAseq_Sox10_Cre_bact_BtR_2dpf_re...,GSM5402430,2dpf,rep_1
...,...,...,...,...,...
TTTGTCATCTTGTACT-1,3,GSM5402430_scRNAseq_Sox10_Cre_bact_BtR_2dpf_re...,GSM5402430,2dpf,rep_1
TTTGTCATCTTTAGGG-1,1,GSM5402430_scRNAseq_Sox10_Cre_bact_BtR_2dpf_re...,GSM5402430,2dpf,rep_1


,gene_id,2
gene_symbol,,
ptpn12,ENSDARG00000102141,Gene Expression
phtf2,ENSDARG00000102123,Gene Expression
...,...,...
TagBFP,TagBFP,Gene Expression
DsRedExpress,DsRedExpress,Gene Expression


'AnnData.raw: No'

'AnnData layers: []'

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [12]:
# .obs column names that should be deleted
drop_obs = []

# .obs column names that should be changed. E.g. "old_name": "New Name"
rename_obs = {}

# .var column names that should be deleted
drop_var = ["2"]

# .var column names that should be changed. E.g. "old_name": "New Name"
rename_var = {}

# Whether to remove the AnnData.raw. Either a boolean or a string to save raw in `adata.layers[<name>]`
allow_raw = False

# adjust layer names. E.g. "old_name": "New Name"
rename_layer = {}

# save adata.X in a layer
keep_X = None

# replace adata.X with another layer
replace_X = None

# layer names that should be kept. 'all' to keep everything or a list of names
keep = 'all'

________

In [13]:
# change obs
obs = adata.obs.copy()

obs.drop(columns=drop_obs, inplace=True)
obs.rename(columns=rename_obs, errors='raise', inplace=True)

# change var
var = adata.var.copy()

var.drop(columns=drop_var, inplace=True)
var.rename(columns=rename_var, errors='raise', inplace=True)

# apply changes to adata
adata.obs = obs
adata.var = var

# edit adata layers
utils.adata.tidy_layers(adata, allow_raw=allow_raw, rename=rename_layer, keep_X=keep_X, replace_X=replace_X, keep=keep)

In [14]:
# map paper cluster to cell types and to spatial components
cell_type_mapping = {
    "0": 'Mesenchyme',
    "1": 'Mesenchyme',
    "2": 'Neuronal',
    "3": 'Mesenchyme',
    "4": 'Mesenchyme',
    "5": 'Neuronal',
    "6": 'Neuronal',
    "7": 'Mesenchyme',
    "8": 'Neuronal',
    "9": 'Mesenchyme',
    "10": 'Neuronal',
    "11": 'Mesenchyme',
    "12": 'Neuronal',
    "13": 'Mesenchyme',
    "14": 'Mesenchyme',
    "15": 'Epithelia',
    "16": 'Neuronal',
    "17": 'Neuronal',
    "18": 'Pigment',
    "19": 'Otic cells',
    "20": 'Neuronal',
    "21": 'Blood',
}

adata.obs["Paper cell type"] = [cell_type_mapping[f"{int(c) - 1}"] for c in adata.obs["paper_cluster"]]

_________

## 5 - Saving the loaded anndata object

In [15]:
# Overview of loaded adata
display(adata)

AnnData object with n_obs × n_vars = 8159 × 32524
    obs: 'paper_cluster', 'filename', 'sample', 'timepoint', 'batch', 'Paper cell type'
    var: 'gene_id'
    uns: 'sctoolbox'

In [16]:
# Saving the data
adata_output = "anndata_1.h5ad"
utils.adata.save_h5ad(adata, adata_output, report=["01_dataset_overview.md", "01_obs.png", "01_var.png"])

[INFO] The adata object was saved to: ../adatas/anndata_1.h5ad


In [17]:
sctoolbox.settings.close_logfile()